In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip
/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt


In [2]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895
词表匹配成功向量：77554/101400
pickle文件生成完成！


In [7]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ===================== Kaggle环境配置 =====================
num_epochs = 12
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
dropout_rate = 0.5
batch_size = 64
labels = 2
lr = 0.001

# 【重点修改】自动规避P100(sm_60)兼容问题
def get_safe_device():
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        # P100 capability = (6,0)，直接强制使用CPU
        if cap[0] == 6:
            logging.warning(f"检测到 {gpu_name}(sm_60)，当前PyTorch不兼容，自动切换至CPU运行")
            return torch.device("cpu")
        else:
            logging.info(f"使用GPU: {gpu_name}")
            return torch.device('cuda:0')
    else:
        return torch.device("cpu")

device = get_safe_device()
use_gpu = device.type == "cuda"

# ===================== TextCNN网络 =====================
class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, dropout_rate, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.use_gpu = use_gpu
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = True

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, k) for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout_rate)
        self.decoder = nn.Linear(num_filter * len(filter_sizes), labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        x = embeddings.permute([0, 2, 1])
        conv_out = [F.relu(conv(x)) for conv in self.convs]
        pool_out = [F.max_pool1d(item, item.size(2)).squeeze(2) for item in conv_out]
        concat = torch.cat(pool_out, dim=1)
        concat = self.dropout(concat)
        outputs = self.decoder(concat)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {' '.join(sys.argv)}")

    logging.info('loading data...')
    pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
    if not os.path.exists(pickle_file):
        raise FileNotFoundError(f"请先运行预处理代码生成文件：{pickle_file}")
    with open(pickle_file, 'rb') as f:
        [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_filter=num_filter,
                       filter_sizes=filter_sizes,
                       dropout_rate=dropout_rate,
                       weight=weight,
                       labels=labels,
                       use_gpu=use_gpu)
    net.to(device)
    loss_function = nn.CrossEntropyLoss()
    # L2正则 weight_decay
    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    # 余弦退火学习率调度
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # ------------------- 早停配置 -------------------
    best_val_acc = 0.0
    patience = 3
    trigger_times = 0
    ckpt_path = "/kaggle/working/best_cnn.pth"
    # 保存最优模型对应的错误样本
    error_samples = []

    for epoch in range(num_epochs):
        start = time.time()
        train_loss = 0.0
        train_acc = 0.0
        n = 0
        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                n += 1
                optimizer.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
                optimizer.step()

                pred = torch.argmax(score.cpu().data, dim=1)
                train_acc += accuracy_score(pred, label.cpu())
                train_loss += loss.item()

                pbar.set_postfix({
                    'loss': f'{train_loss / n:.4f}',
                    'acc': f'{train_acc / n:.4f}'
                })
                pbar.update(1)

        # 验证阶段
        val_losses = 0.0
        val_acc = 0.0
        m = 0
        net.eval()
        temp_errors = []
        with torch.no_grad():
            with tqdm(total=len(val_iter), desc=f'Epoch {epoch} Val') as pbar:
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    pred = torch.argmax(val_score.cpu().data, dim=1)
                    true = val_label.cpu()
                    val_acc += accuracy_score(pred, true)
                    val_losses += val_loss.item()

                    # 收集预测错误样本
                    wrong_mask = (pred != true)
                    wrong_idx = torch.nonzero(wrong_mask).squeeze(1)
                    for idx in wrong_idx:
                        seq_ids = val_feature[idx].cpu().tolist()
                        t = true[idx].item()
                        p = pred[idx].item()
                        temp_errors.append({
                            "seq_ids": seq_ids,
                            "true_label": t,
                            "pred_label": p
                        })
                    pbar.update(1)

        end = time.time()
        runtime = end - start
        current_val_acc = val_acc / m
        logging.info(
            f"Epoch {epoch} | Train Loss:{train_loss/n:.4f} Train Acc:{train_acc/n:.4f} "
            f"Val Loss:{val_losses/m:.4f} Val Acc:{current_val_acc:.4f} Time:{runtime:.2f}s"
        )

        # 早停判断 & 保存最优模型 + 缓存此时错误样本
        if current_val_acc > best_val_acc:
            best_val_acc = current_val_acc
            trigger_times = 0
            torch.save(net.state_dict(), ckpt_path)
            error_samples = temp_errors.copy()
            logging.info(f"Save best model, val acc = {best_val_acc:.4f}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                logging.info(f"Early Stop! Best val acc:{best_val_acc:.4f}")
                break
        scheduler.step()

    # ----------------【输出3条错误案例，用于错误分析】----------------
    logging.info("\n========== 验证集错误样例（最多打印3条） ==========")
    show_num = min(3, len(error_samples))
    for i in range(show_num):
        item = error_samples[i]
        ids_list = item["seq_ids"]
        # 将index序列还原成单词（去掉padding 0）
        words = [idx_to_word[idx] for idx in ids_list if idx != 0]
        print(f"\n【错误案例{i+1}】")
        print(f"真实标签：{item['true_label']}，预测标签：{item['pred_label']}")
        print(f"文本：{' '.join(words)}")

    # ---------------- 预测阶段：加载最优权重 ----------------
    net.load_state_dict(torch.load(ckpt_path, map_location=device))
    net.eval()
    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    # 生成提交文件
    TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("/kaggle/working/cnn.csv", index=False, quoting=3)
    logging.info('result saved!')

2026-08-14 07:49:24,885: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-c8522ec4-cd3e-4e18-9075-43883389112e.json
2026-08-14 07:49:24,886: INFO: loading data...
2026-08-14 07:49:26,163: INFO: data loaded!
Epoch 0 Val: 100%|██████████| 79/79 [00:33<00:00,  2.33it/s]
2026-08-14 07:58:51,966: INFO: Epoch 0 | Train Loss:0.4109 Train Acc:0.8048 Val Loss:0.3029 Val Acc:0.8744 Time:565.77s
2026-08-14 07:58:52,215: INFO: Save best model, val acc = 0.8744
Epoch 1 Val: 100%|██████████| 79/79 [00:35<00:00,  2.25it/s]
2026-08-14 08:08:01,105: INFO: Epoch 1 | Train Loss:0.2933 Train Acc:0.8763 Val Loss:0.2902 Val Acc:0.8795 Time:548.89s
2026-08-14 08:08:01,367: INFO: Save best model, val acc = 0.8795
Epoch 2 Val: 100%|██████████| 79/79 [00:35<00:00,  2.25it/s]
2026-08-14 08:17:19,886: INFO: Epoch 2 | Train Loss:0.2354 Train Acc:0.9052 Val Loss:0.2734 Val Acc:0.8928 Time:558.52s
2026-08-14 08:17:20,138: INFO: Save best mode


【错误案例1】
真实标签：0，预测标签：1
文本：natural born killers cinema cut r director s cut nc it s an unusual oliver stone picture but when i read he was on drugs during the filming i needed no further explanation natural born killers is a risky mad all out film making that we do not get very often strange psychotic artistic pictures natural born killers is basically the story of how two mass killers were popularised and glorified by the media there is a great scene where an interviewer questions some teenagers about mickey and mallory and the teenager says murder is wrong but if i was a mass murderer i d be mickey and mallory mickey describes this with a situation of frankenstein the monster and dr frankenstein dr frankenstein is the media who has turned them into these monstrous killersmost oliver stone films examine the flaws of the america the country that the director loves and admires i guess natural born killers is about the effect of mass media technology and how obsessive as a nation american

Prediction: 100%|██████████| 391/391 [02:50<00:00,  2.30it/s]
2026-08-14 08:59:14,156: INFO: result saved!


In [3]:
import logging
import os
import sys
import pickle
import time
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

# ===================== Kaggle环境超参配置 =====================
num_epochs = 15
embed_size = 300
num_filter = 128
filter_sizes = [3, 4, 5]
dropout_rate = 0.4
batch_size = 64
lr = 0.001
freeze_epoch = 4       # 前4轮冻结GloVe
weight_decay = 1e-4
patience = 3
# ==============================================================

# 自动规避P100(sm_60)兼容问题
def get_safe_device():
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        if cap[0] == 6:
            logging.warning(f"检测到 {gpu_name}(sm_60)，当前PyTorch不兼容，自动切换至CPU运行")
            return torch.device("cpu")
        else:
            logging.info(f"使用GPU: {gpu_name}")
            return torch.device('cuda:0')
    else:
        return torch.device("cpu")

device = get_safe_device()
use_gpu = device.type == "cuda"

# ===================== TextCNN网络 =====================
class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_filter, filter_sizes, dropout_rate, weight, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.use_gpu = use_gpu
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False  # 初始冻结，训练代码动态切换

        self.dropout = nn.Dropout(dropout_rate)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_size, num_filter, k) for k in filter_sizes
        ])
        self.decoder = nn.Linear(num_filter * len(filter_sizes), 1)  # 输出1维logits

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = self.dropout(embeddings)          # dropout放在embedding之后
        x = embeddings.permute([0, 2, 1])              # [B, seq_len, dim] -> [B, dim, seq_len]
        conv_out = [F.relu(conv(x)) for conv in self.convs]
        pool_out = [F.max_pool1d(item, item.size(2)).squeeze(2) for item in conv_out]
        concat = torch.cat(pool_out, dim=1)
        outputs = self.decoder(concat)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {' '.join(sys.argv)}")

    logging.info('loading data...')
    pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
    if not os.path.exists(pickle_file):
        raise FileNotFoundError(f"请先运行预处理代码生成文件：{pickle_file}")
    with open(pickle_file, 'rb') as f:
        [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab] = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_filter=num_filter,
                       filter_sizes=filter_sizes,
                       dropout_rate=dropout_rate,
                       weight=weight,
                       use_gpu=use_gpu)
    net.to(device)

    loss_function = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    # 基于验证AUC衰减学习率
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-5)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    # ------------------- 早停配置（以AUC为指标） -------------------
    best_val_auc = 0.0
    trigger_times = 0
    ckpt_path = "/kaggle/working/best_cnn.pth"
    error_samples = []

    for epoch in range(num_epochs):
        start = time.time()
        train_loss = 0.0
        n = 0
        net.train()

        # 两阶段控制embedding是否可训练
        if epoch < freeze_epoch:
            for param in net.embedding.parameters():
                param.requires_grad = False
        else:
            for param in net.embedding.parameters():
                param.requires_grad = True

        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                n += 1
                optimizer.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label.unsqueeze(1).float())
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
                optimizer.step()

                train_loss += loss.item()
                pbar.set_postfix({'loss': f'{train_loss / n:.4f}'})
                pbar.update(1)

        # 验证阶段：计算AUC + Acc
        val_losses = 0.0
        val_y_true = []
        val_y_prob = []
        temp_errors = []
        m = 0
        net.eval()
        with torch.no_grad():
            with tqdm(total=len(val_iter), desc=f'Epoch {epoch} Val') as pbar:
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label.unsqueeze(1).float())
                    val_losses += val_loss.item()

                    prob = torch.sigmoid(val_score).squeeze(1).cpu()
                    pred = (prob > 0.5).long()
                    true = val_label.cpu()

                    val_y_true.extend(true.numpy().tolist())
                    val_y_prob.extend(prob.numpy().tolist())

                    # 收集错误样本
                    wrong_mask = (pred != true)
                    wrong_idx = torch.nonzero(wrong_mask).squeeze(1)
                    for idx in wrong_idx:
                        seq_ids = val_feature[idx].cpu().tolist()
                        t = true[idx].item()
                        p = pred[idx].item()
                        temp_errors.append({
                            "seq_ids": seq_ids,
                            "true_label": t,
                            "pred_label": p
                        })
                    pbar.update(1)

        end = time.time()
        runtime = end - start
        current_val_loss = val_losses / m
        current_val_auc = roc_auc_score(val_y_true, val_y_prob)
        current_val_acc = accuracy_score(val_y_true, (np.array(val_y_prob) > 0.5))

        logging.info(
            f"Epoch {epoch} | Train Loss:{train_loss/n:.4f} "
            f"Val Loss:{current_val_loss:.4f} Val Acc:{current_val_acc:.4f} Val AUC:{current_val_auc:.4f} Time:{runtime:.2f}s"
        )

        # 早停 & 保存最优模型（依据AUC）
        if current_val_auc > best_val_auc:
            best_val_auc = current_val_auc
            trigger_times = 0
            torch.save(net.state_dict(), ckpt_path)
            error_samples = temp_errors.copy()
            logging.info(f"Save best model, val auc = {best_val_auc:.4f}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                logging.info(f"Early Stop! Best val AUC:{best_val_auc:.4f}")
                break
        scheduler.step(current_val_auc)

    # ----------------【输出3条错误案例，用于错误分析】----------------
    logging.info("\n========== 验证集错误样例（最多打印3条） ==========")
    show_num = min(3, len(error_samples))
    for i in range(show_num):
        item = error_samples[i]
        ids_list = item["seq_ids"]
        words = [idx_to_word[idx] for idx in ids_list if idx != 0]
        print(f"\n【错误案例{i+1}】")
        print(f"真实标签：{item['true_label']}，预测标签：{item['pred_label']}")
        print(f"文本：{' '.join(words)}")

    # ---------------- 预测阶段：加载最优权重 ----------------
    net.load_state_dict(torch.load(ckpt_path, map_location=device))
    net.eval()
    test_pred_prob = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                prob = torch.sigmoid(test_score).squeeze(1).cpu().numpy().tolist()
                test_pred_prob.extend(prob)
                pbar.update(1)

    # 生成Kaggle提交csv（阈值0.5转为0/1）
    TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
    test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)
    test["sentiment"] = (np.array(test_pred_prob) > 0.5).astype(int)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test["sentiment"]})
    result_output.to_csv("/kaggle/working/cnn.csv", index=False, quoting=3)
    logging.info('result saved!')


INFO:colab_kernel_launcher.py:running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-a7b3581d-96a8-4037-9a40-41b5a0ea552c.json
INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Val: 100%|██████████| 79/79 [00:30<00:00,  2.63it/s]
INFO:root:Epoch 0 | Train Loss:0.4207 Val Loss:0.3017 Val Acc:0.8766 Val AUC:0.9445 Time:357.46s
INFO:root:Save best model, val auc = 0.9445
Epoch 1 Val: 100%|██████████| 79/79 [00:26<00:00,  2.99it/s]
INFO:root:Epoch 1 | Train Loss:0.2925 Val Loss:0.2749 Val Acc:0.8866 Val AUC:0.9548 Time:324.05s
INFO:root:Save best model, val auc = 0.9548
Epoch 2 Val: 100%|██████████| 79/79 [00:30<00:00,  2.63it/s]
INFO:root:Epoch 2 | Train Loss:0.2474 Val Loss:0.2614 Val Acc:0.8926 Val AUC:0.9582 Time:332.83s
INFO:root:Save best model, val auc = 0.9582
Epoch 3 Val: 100%|██████████| 79/79 [00:30<00:00,  2.60it/s]
INFO:root:Epoch 3 | Train Loss:0.2155 Val Loss:0.2533 Val Acc:0.8990 Val AUC:0.9611 Time:350.0


【错误案例1】
真实标签：0，预测标签：1
文本：i vaguely remember ben from my sci fi fandom days of the s i was doing several interviews bios of obscure actors actresses most notably ben actress fay spain and jody fair who played angela in s the young savages ben was one of the people at a low key sci fi con in chicago about when i had a nice chat with him and his career and life all these were published in some now long forgotten fanzine of the day wish i still had copies of those interviews but time marches on and any of those people surely wouldn t remember me at all so many years later ben was a really nice fellow ekeing out a living the cons of those days didn t even pay their guest unless of course they were big name stars and even then the pay was a couple hundred dollars at most good to know ben s still alive kicking how bout a remake of creature but years older ugly then uglier now

【错误案例2】
真实标签：0，预测标签：1
文本：natural born killers cinema cut r director s cut nc it s an unusual oliver stone picture bu

Prediction: 100%|██████████| 391/391 [02:22<00:00,  2.74it/s]
INFO:root:result saved!
